
# 5. Production Prediction

This notebook demonstrates how the trained machine learning model predicts actual production quantity (`actual_qty`) for a new shoe production order.

The selected Gradient Boosting regression pipeline is saved as `models/production_model.joblib`. It contains the fitted preprocessing transformations and the trained model required to generate predictions.

The prediction workflow consists of four steps:

1. Load the saved model pipeline.
2. Prepare a production order using the required input features.
3. Recreate the engineered features used during model training.
4. Generate and interpret the predicted actual production quantity.

The model predicts `actual_qty`, representing the estimated number of shoes produced under the supplied production conditions. The prediction is an estimate based on historical data, not a guaranteed production outcome.

**Input:** The saved model pipeline and a DataFrame containing the required production inputs.

**Output:** A predicted actual production quantity, rounded to a whole number of shoes.

The prediction can subsequently support prescriptive analytics by comparing estimated production output with a planned target. Any resource recommendations require a separate analysis that considers operational constraints.


## 5.1 Import Required Libraries

Pandas is used to organise the information for a new production order into a DataFrame. Joblib is used to load the saved machine learning pipeline.

The saved pipeline includes both the preprocessing transformations and the trained regression model. Loading the complete pipeline ensures that the input data is processed consistently with the data used during model training.

Using the saved pipeline avoids the need to retrain the model whenever a new production prediction is required.

In [11]:

import pandas as pd
import joblib
from pathlib import Path
import numpy as np

## 5.2 Define the Production Prediction Function

The `predict_production()` function provides a reusable way to generate production forecasts for new input records.

The function performs three main operations.

### 5.2.1 Load the Saved Model

The saved pipeline is loaded from `production_model.joblib` using Joblib.

The pipeline contains the fitted preprocessing steps and the selected Gradient Boosting regression model. This allows new production records to pass through the same transformations used during model development.

### 5.2.2 Recreate the Engineered Features

Two engineered features are calculated from the supplied input variables:

- `machine_worker_ratio = machine_count / no_of_workers`
- `avg_hours_per_worker = working_hours / no_of_workers`

These features represent machines per worker and average recorded working hours per worker, respectively.

They must be calculated consistently with the feature engineering performed during model training. If the formulas or feature names differ, the prediction pipeline may receive inputs that do not match the structure expected by the trained model.

The function creates a copy of the input DataFrame before adding the features. This prevents the original input DataFrame from being modified by these operations.

The calculations require `no_of_workers` to be greater than zero. The supplied input should also contain valid values for the required columns.

### 5.2.3 Generate the Prediction

The prepared DataFrame is passed to the saved pipeline, which applies the fitted preprocessing transformations and generates a prediction for each input record.

The resulting predictions are rounded to whole numbers and converted to integers because the output represents a quantity of shoes.

The function can process multiple production orders in a single DataFrame. It returns one predicted actual production quantity for each row.

The output is an estimate of expected production, not a guarantee of the quantity that will ultimately be produced.

In [12]:

def predict_production(
    new_data,
    model_path="../models/production_model.joblib"
):
    """
    Predict actual production quantity for one or more production orders.

    Parameters
    ----------
    new_data : pandas.DataFrame
        DataFrame containing the 14 original model input columns.
    model_path : str or Path
        Path to the saved fitted model pipeline.

    Returns
    -------
    numpy.ndarray
        Predicted actual production quantities rounded to whole numbers.
    """

    # Check the input type
    if not isinstance(new_data, pd.DataFrame):
        raise TypeError("new_data must be a pandas DataFrame.")

    if new_data.empty:
        raise ValueError("new_data contains no production orders.")

    # Load the saved model pipeline
    model_path = Path(model_path)

    if not model_path.exists():
        raise FileNotFoundError(
            f"Saved model not found: {model_path.resolve()}. "
            "Run Notebook 4 first and confirm the model was saved "
            "in the models folder."
        )

    pipeline = joblib.load(model_path)

    # Required original input columns
    required_cols = [
        "financial_month",
        "week_no",
        "week_of_month",
        "factory_unit",
        "product_family",
        "item_name",
        "size_id",
        "grade",
        "rate",
        "no_of_workers",
        "working_hours",
        "machine_count",
        "order_category",
        "bottleneck_flag"
    ]

    missing_cols = [
        col for col in required_cols if col not in new_data.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required input columns: {missing_cols}"
        )

    # Use only the expected original input columns
    df_pred = new_data[required_cols].copy()

    # Check for missing input values
    if df_pred.isna().any().any():
        raise ValueError(
            "Missing values found in the input data. "
            "Please provide values for all required columns."
        )

    # Validate resource inputs
    if (df_pred["no_of_workers"] <= 0).any():
        raise ValueError("no_of_workers must be greater than zero.")

    if (df_pred["working_hours"] < 0).any():
        raise ValueError("working_hours cannot be negative.")

    if (df_pred["machine_count"] < 0).any():
        raise ValueError("machine_count cannot be negative.")

    if (df_pred["rate"] < 0).any():
        raise ValueError("rate cannot be negative.")

    if not df_pred["bottleneck_flag"].isin([0, 1, True, False]).all():
        raise ValueError("bottleneck_flag must contain only 0 or 1.")

    # Ensure numeric columns contain valid numerical values
    numeric_cols = [
        "week_no",
        "week_of_month",
        "size_id",
        "rate",
        "no_of_workers",
        "working_hours",
        "machine_count"
    ]

    for col in numeric_cols:
        df_pred[col] = pd.to_numeric(df_pred[col], errors="raise")

    if not np.isfinite(df_pred[numeric_cols].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite values found in numerical inputs.")

    # Recreate the engineered features used during training
    df_pred["machine_worker_ratio"] = (
        df_pred["machine_count"] / df_pred["no_of_workers"]
    )

    df_pred["avg_hours_per_worker"] = (
        df_pred["working_hours"] / df_pred["no_of_workers"]
    )

    # Generate predictions
    predictions = pipeline.predict(df_pred)

    # Round predicted quantities to whole shoes
    return np.rint(predictions).astype(int)


## 5.3 Prepare a New Production Order

A new production order is represented as a Pandas DataFrame containing the 14 input columns required by the prediction function.

These columns describe the production period, factory unit, product characteristics, workforce, working hours, available machines, order category, and known bottleneck conditions.

| Input column | Purpose |
|---|---|
| `financial_month` | Represents the financial month associated with the production order. |
| `week_no` | Indicates the week number used to describe the production period. |
| `week_of_month` | Represents the position of the week within the month. |
| `factory_unit` | Identifies the factory unit responsible for production. |
| `product_family` | Identifies the product family, such as Ladies or Gents. |
| `item_name` | Identifies the specific shoe item. |
| `size_id` | Specifies the shoe size. |
| `grade` | Represents the product grade. |
| `rate` | Provides the recorded rate value used by the model. |
| `no_of_workers` | Specifies the number of workers assigned. |
| `working_hours` | Provides the recorded working hours for the production order. |
| `machine_count` | Specifies the number of machines available. |
| `order_category` | Identifies the order category, such as School, Forces/Police, or Others. |
| `bottleneck_flag` | Indicates a known machine fault or constraint using 0 or 1. |

The values supplied should describe the production conditions expected for the order being forecast.

### 5.3.1 Why Are These Inputs Required?

The model learned relationships between the input features and `actual_qty` during training. Providing the same types of information allows the saved pipeline to apply those learned relationships to a new production scenario.

For example, changing the number of workers, working hours, or machine count changes the information supplied to the model and may therefore change the predicted output. The direction and size of the change must be determined from the model's prediction rather than assumed in advance.

The input should contain valid values that follow the formats expected by the trained pipeline. The engineered ratio columns do not need to be entered manually because the prediction function calculates them from the supplied resource variables.

### 5.3.2 Columns Not Required for Prediction

The following columns are not required by this prediction function:

- `planned_qty`
- `production_gap_qty`
- `total_duration_per_shoe_mins`
- `week_start_date`
- `week_end_date`

These columns are excluded from the model inputs used in this notebook. In particular, `production_gap_qty` depends on actual production quantity and would reveal information about the target if supplied as a predictor.

The date columns are also not passed into the model. The prediction therefore depends on the supplied production features rather than directly using the calendar dates.


### 5.3.3 Unseen Categories

A new production order may contain a category or item that was not present in the training data.

The one-hot encoder is configured to ignore unknown categories. The behaviour of the target encoder for unseen item names depends on its configuration and the version of the preprocessing pipeline used during training.

Although the pipeline may be able to generate predictions for previously unseen categories, those predictions may be less reliable when the relevant category or item has little or no representation in the training data. Predictions for unseen categories should therefore be interpreted cautiously.


### 5.3.4 Importance of Input Validity

The prediction function assumes that the required columns exist and contain valid values.

In particular, `no_of_workers` must be greater than zero because it is used as the denominator in both engineered features. The `bottleneck_flag` should be supplied as 0 or 1 and should only indicate a condition that is already known when the prediction is made.

The model's usefulness depends on the supplied inputs accurately representing the conditions under which the production order will be carried out.

### 5.3.5 Example Production Order

A sample production order is created to demonstrate how the prediction function can be used.

The example specifies the financial month, week, factory unit, product family, item, size, grade, rate, workforce size, working hours, machine count, order category, and bottleneck status.

These values provide a hypothetical production scenario for testing the prediction workflow. They should not be interpreted as a verified production plan or as a statement about the actual resources assigned to a real order.

The DataFrame is displayed so that the supplied values can be checked before generating the prediction.

For a different scenario, the input values can be changed and the prediction function can be executed again. This makes it possible to examine how alternative resource configurations are reflected in the model's forecasts.

In [13]:
# Load the original dataset to get the last 5 rows
# Assuming the dataset was loaded in previous notebooks, we need to load it here.
# Since the dataset path isn't explicitly in this notebook, I'll use a common path 
# or look for where it might be. Based on common project structures, it's likely in 'data/'.
import pandas as pd

# Attempt to load the dataset. I will use the most likely path.
# If this fails, the user might need to provide the path.
try:
    df_full = pd.read_csv('../data/processed/final_features.csv') # Typical path
except FileNotFoundError:
    # Fallback: search for the csv file if the typical path fails
    import glob
    csv_files = glob.glob('../data/*.csv')
    if csv_files:
        df_full = pd.read_csv(csv_files[0])
    else:
        raise FileNotFoundError("Could not find the dataset CSV file in ../data/")

# The model requires 14 specific input columns
required_cols = [
    "financial_month",
    "week_no",
    "week_of_month",
    "factory_unit",
    "product_family",
    "item_name",
    "size_id",
    "grade",
    "rate",
    "no_of_workers",
    "working_hours",
    "machine_count",
    "order_category",
    "bottleneck_flag"
]

# Get the last 5 rows and only the required columns
new_order = df_full[required_cols].tail(5).reset_index(drop=True)
new_order

,financial_month,week_no,week_of_month,factory_unit,product_family,item_name,size_id,grade,rate,no_of_workers,working_hours,machine_count,order_category,bottleneck_flag
0,Mar,52,5,Factory Unit 01,Gents,DIYANA - L - BROWN,8,A,4954.61,12,436.74,4,Others,1
1,Mar,52,5,Factory Unit 01,Gents,EX/DE/11 - L - MULTI,12,A,5994.27,8,328.67,5,Others,1
2,Mar,52,5,Factory Unit 01,Gents,GB 812 - L - BROWN,11,A,4362.43,18,729.04,4,Others,0
3,Mar,52,5,Factory Unit 01,Ladies,DALRINE - L - BROWN,6,A,2980.86,9,343.44,4,Others,0
4,Mar,52,5,Factory Unit 02 - LGF,Gents,TRUE MOCCACINO - L - BLACK,8,A,4517.57,16,637.79,3,Others,0


## 5.4 Generate the Production Prediction

The prepared production order is passed to `predict_production()`.

The function loads the saved pipeline, calculates the two engineered resource features, applies the fitted preprocessing transformations, and generates a prediction for `actual_qty`.

The predicted value is returned as a whole number because the quantity of shoes is expressed as an integer.

The output represents the model's estimate of the actual production quantity under the supplied conditions. It is a predictive estimate based on patterns learned from historical records rather than a direct measurement of future production.

The prediction can be used as an input to the next stage of the project, where expected production output can be compared with a planned quantity to support production planning.

In [14]:
predicted_qtys = predict_production(new_order)

# Create a results DataFrame for better visibility
results = new_order.copy()
results['Predicted_Actual_Qty'] = predicted_qtys

print(f"Predictions for the last 5 rows:\n\n{results[['item_name', 'Predicted_Actual_Qty']]}")

Predictions for the last 5 rows:

                    item_name  Predicted_Actual_Qty
0          DIYANA - L - BROWN                    26
1        EX/DE/11 - L - MULTI                    25
2          GB 812 - L - BROWN                    35
3         DALRINE - L - BROWN                    27
4  TRUE MOCCACINO - L - BLACK                    29



## 5.5 Interpret the Prediction

The printed output shows the predicted actual production quantity for the production order supplied to the prediction function.

The predicted quantity represents the model's estimate of the number of shoes that may be produced under the specified production conditions. The value is rounded to a whole number because production quantity is expressed as a count of shoes.

### 5.5.1 Model Performance Context

The selected Gradient Boosting model achieved the following results on the chronological test set in Notebook 4:

- **Test R² = 0.797:** The model explained approximately 79.7% of the variation in actual production quantities relative to a test-set mean baseline.
- **Test MAE = 4.504:** The average absolute prediction error was approximately 4.50 shoes per test record.
- **Test RMSE = 5.900:** The root mean squared error was approximately 5.90 shoes per test record, with larger errors receiving greater weight.

These metrics describe the model's performance across the held-out test records. They do not establish the exact error for this individual production order, and the actual production quantity may differ from the prediction.

### 5.5.2 Examine Alternative Production Scenarios

Alternative scenarios can be created by changing inputs such as the number of workers, working hours, machine count, or bottleneck status.

The prediction function can then be executed again to compare the model's estimated output under different input configurations. When investigating one variable at a time, the other inputs should remain constant.

These comparisons show how the trained model responds to alternative scenarios; they do not prove that changing a particular resource will cause the predicted change in real production. Scenarios should remain operationally realistic and respect factory constraints.

The model's predictions can support production planning, but resource allocation decisions should also consider operational knowledge, capacity constraints, and validation against actual production outcomes.

